<a href="https://colab.research.google.com/github/vnsantos/projeto_mrp_costing/blob/main/PROJETO_MRP_COSTING_(FINAL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
ARQUIVO_EXCEL = list(uploaded.keys())[0]

xls = pd.ExcelFile(ARQUIVO_EXCEL)

aba_listagem = next(s for s in xls.sheet_names if 'listagem' in s.lower())
aba_receita  = next(s for s in xls.sheet_names if 'receita' in s.lower())

df_listagem_raw = pd.read_excel(ARQUIVO_EXCEL, sheet_name=aba_listagem)
df_receita_raw  = pd.read_excel(ARQUIVO_EXCEL, sheet_name=aba_receita)

df_listagem = pd.DataFrame({
    "cia": df_listagem_raw.iloc[:, 1],
    "cardapio": df_listagem_raw.iloc[:, 3],
    "servico": df_listagem_raw.iloc[:, 5],
    "class_": df_listagem_raw.iloc[:, 8],
    "codigo_inflightor": df_listagem_raw.iloc[:, 9],
    "produto": df_listagem_raw.iloc[:, 10],
    "classe": df_listagem_raw.iloc[:, 11],
    "ciclo": df_listagem_raw.iloc[:, 12],
    "pct": df_listagem_raw.iloc[:, 13],
    "qtd": df_listagem_raw.iloc[:, 14],
    "un": df_listagem_raw.iloc[:, 15],
    "custol": df_listagem_raw.iloc[:, 16],
    "representatividade": df_listagem_raw.iloc[:, 17],
    "base_custo": df_listagem_raw.iloc[:, 20]
}).dropna(subset=["produto"])

df_receita = pd.DataFrame({
    "receita_principal": df_receita_raw.iloc[:, 10],
    "base": df_receita_raw.iloc[:, 0],
    "cod_menu": df_receita_raw.iloc[:, 1],
    "receita": df_receita_raw.iloc[:, 11],
    "class_": df_receita_raw.iloc[:, 8],
    "codigo": df_receita_raw.iloc[:, 9],
    "yield_": df_receita_raw.iloc[:, 12],
    "quantidade": df_receita_raw.iloc[:, 13],
    "qtd_unt": df_receita_raw.iloc[:, 14],
    "unidade": df_receita_raw.iloc[:, 15],
    "rend_liq": df_receita_raw.iloc[:, 16],
    "rend_kg": df_receita_raw.iloc[:, 17],
    "custo_unit": df_receita_raw.iloc[:, 18],
    "custo_total": df_receita_raw.iloc[:, 19],
    "custo": df_receita_raw.iloc[:, 20]
}).dropna(subset=["receita_principal", "receita"])


def convert_cols(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )

convert_cols(df_listagem, ["qtd", "custol", "pct", "representatividade"])
convert_cols(df_receita, [
    "yield_", "quantidade", "qtd_unt",
    "rend_liq", "rend_kg",
    "custo_unit", "custo_total", "custo"
])


df_listagem["produto_norm"] = df_listagem["produto"].astype(str).str.strip().str.upper()
df_receita["principal_norm"] = df_receita["receita_principal"].astype(str).str.strip().str.upper()
df_receita["receita_norm"]   = df_receita["receita"].astype(str).str.strip().str.upper()

receita_dict = dict(tuple(df_receita.groupby("principal_norm")))


resultado = []
append_result = resultado.append

def explodir_arvore(receita_norm, row, visitadas):
    stack = [receita_norm]

    while stack:
        atual = stack.pop()

        if atual in visitadas:
            continue

        visitadas.add(atual)

        itens = receita_dict.get(atual)
        if itens is None:
            continue

        for item in itens.itertuples(index=False):

            append_result((
                item.base,item.cod_menu,row.base_custo,row.cia, row.cardapio, row.servico,
                item.class_, item.codigo, item.receita,
                row.classe, row.ciclo, row.qtd, row.pct,
                row.un, row.custol,
                item.yield_, item.quantidade, item.qtd_unt, item.unidade,
                item.rend_liq, item.rend_kg,
                item.custo_unit, item.custo_total, item.custo
            ))

            if item.class_ == "S":
                stack.append(item.receita_norm)


for row in df_listagem.itertuples(index=False):

    append_result((
        None, None, row.base_custo,row.cia, row.cardapio, row.servico,
        row.class_, row.codigo_inflightor, row.produto,
        row.classe, row.ciclo, row.qtd, row.pct,
        row.un, row.custol,

        None, None, None, None,
        None, None, None, None, None
    ))

    if row.class_ != "P":
        explodir_arvore(row.produto_norm, row, set())


colunas = [
    "Base","Cod Menu Grid","Base de custo",
    "Cia","Cardapio","Serviço","Class","Codigo inflightor","Produto e receita",
    "Classe","Ciclo","Qtd","%","UN","CustoL","Yield","Quantidade","Qtd Unt",
    "Unidade","Rendimento Liquido","Rendimento KG",
    "Custo Unit","Custo Total","Custo"
]

df_final = pd.DataFrame(resultado, columns=colunas)


mask_r = df_final['Class'] == 'R'

df_final['Qtd_ref'] = df_final['Qtd'].where(mask_r).ffill()

df_final['Custo Unitario'] = df_final['Custo'] * df_final['Qtd_ref']

mask_p = (df_final['Yield'].isna()) & (df_final['Class'] == 'P')
df_final.loc[mask_p, 'Custo Unitario'] = df_final['CustoL'] * df_final['Qtd_ref']

df_final['Unid Unit'] = df_final['Quantidade'] * df_final['Qtd_ref']

df_final[['Cardapio','Base','Cod Menu Grid']] = df_final[
    ['Cardapio','Base','Cod Menu Grid']
].ffill()


saida = "LIST_REC_CONSOLIDADO_FINAL.xlsx"
df_final.to_excel(saida, index=False)

files.download(saida)

Saving SALAVIP.xlsx to SALAVIP.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>